# 08 — Explicabilité : SHAP & LIME
**Membre 3 — XAI Expert**

Ce notebook couvre :
1. SHAP — 3 niveaux d'explication (global, local, interaction)
2. LIME — validation croisée avec SHAP
3. Analyse sur 3 profils clients (faible risque, risque moyen, haut risque)
4. Contexte légal : RGPD Article 22, AI Act

> **Prérequis** : avoir exécuté `03_pipeline.ipynb` (Membre 1) et `04_classification.ipynb` (Membre 2)  
> Le modèle LightGBM optimisé doit être sauvegardé dans `../models/lgbm_model.pkl`


In [1]:
# ── Installations ────────────────────────────────────────────
# !pip install shap lime lightgbm scikit-learn pandas numpy matplotlib seaborn

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os

# SHAP
import shap

# LIME
from lime import lime_tabular

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

print("✅ Imports OK")
print(f"SHAP version : {shap.__version__}")


✅ Imports OK
SHAP version : 0.51.0


## 1. Chargement des données et du modèle

In [3]:
# ── Chargement des données nettoyées (sortie Membre 1) ───────
DATA_PATH = "../data/processed/lending_clean.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Fichier introuvable : {DATA_PATH}\n"
        "Assurez-vous d'avoir exécuté 01_cleaning.ipynb et 02_feature_eng.ipynb"
    )

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Shape : {df.shape}")
print(f"Distribution target :\n{df['default'].value_counts(normalize=True).round(3)}")


Shape : (1345310, 115)


KeyError: 'default'

In [4]:
# ── Features utilisées pour la modélisation ──────────────────
# (doit correspondre exactement à ce qui a été utilisé dans 04_classification.ipynb)
FEATURES = [
    'loan_amnt', 'term', 'int_rate', 'installment', 'grade',
    'emp_length', 'annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths',
    'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
    'loan_to_income', 'installment_to_income', 'credit_history_years',
    'int_rate_x_term', 'dti_x_revol', 'purpose', 'home_ownership', 'addr_state'
]
TARGET = 'default'

# Garder uniquement les colonnes disponibles
available_features = [f for f in FEATURES if f in df.columns]
print(f"Features disponibles : {len(available_features)}/{len(FEATURES)}")

X = df[available_features].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train : {X_train.shape}, Test : {X_test.shape}")


Features disponibles : 18/23


KeyError: 'default'

In [ ]:
# ── Chargement du modèle LightGBM (Membre 2) ─────────────────
MODEL_PATH = "../models/lgbm_model.pkl"

if os.path.exists(MODEL_PATH):
    model = joblib.load(MODEL_PATH)
    print(f"✅ Modèle chargé : {MODEL_PATH}")
    print(f"Type : {type(model)}")
else:
    print("⚠️  Modèle non trouvé — entraînement d'un modèle de démonstration...")
    import lightgbm as lgb
    from sklearn.preprocessing import OrdinalEncoder
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline

    numeric_cols = X_train.select_dtypes(include='number').columns.tolist()
    cat_cols = X_train.select_dtypes(include='object').columns.tolist()

    from sklearn.pipeline import Pipeline as skPipeline
    from sklearn.preprocessing import StandardScaler

    preprocessor = ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer()), ('sc', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                          ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), cat_cols)
    ])

    model = skPipeline([
        ('prep', preprocessor),
        ('clf', lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05,
                                    num_leaves=63, random_state=42, verbose=-1))
    ])
    model.fit(X_train, y_train)
    os.makedirs("../models", exist_ok=True)
    joblib.dump(model, MODEL_PATH)
    print("✅ Modèle de démonstration entraîné et sauvegardé")


## 2. SHAP — Préparation de l'explainer

In [ ]:
# ── Extraction du modèle LightGBM depuis le pipeline ─────────
from sklearn.pipeline import Pipeline as skPipeline

if hasattr(model, 'named_steps'):
    # C'est un pipeline sklearn
    lgbm_clf = model.named_steps.get('clf', model.named_steps.get('lgbm', list(model.named_steps.values())[-1]))
    X_test_transformed = model[:-1].transform(X_test)
    X_train_transformed = model[:-1].transform(X_train)
    
    # Récupérer les noms de features après transformation
    try:
        feature_names = model[:-1].get_feature_names_out()
    except:
        feature_names = [f"feature_{i}" for i in range(X_test_transformed.shape[1])]
else:
    lgbm_clf = model
    X_test_transformed = X_test
    X_train_transformed = X_train
    feature_names = available_features

print(f"Shape données transformées : {X_test_transformed.shape}")
print(f"Modèle : {type(lgbm_clf).__name__}")


In [ ]:
# ── Création de l'explainer SHAP ──────────────────────────────
# TreeExplainer est optimisé pour LightGBM / XGBoost / Random Forest
explainer = shap.TreeExplainer(lgbm_clf)

# Calculer les SHAP values sur un échantillon (pour performance)
SAMPLE_SIZE = 2000
idx_sample = np.random.RandomState(42).choice(len(X_test_transformed), 
                                               min(SAMPLE_SIZE, len(X_test_transformed)), 
                                               replace=False)

X_sample = X_test_transformed[idx_sample] if hasattr(X_test_transformed, '__getitem__')            else X_test_transformed.iloc[idx_sample]

shap_values = explainer.shap_values(X_sample)

# Pour la classification binaire, LightGBM retourne parfois une liste [class0, class1]
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # On garde la classe 1 (défaut)

print(f"SHAP values shape : {shap_values.shape}")
print("✅ SHAP values calculées")


## 3. SHAP — Vue Globale : Beeswarm Plot

In [ ]:
# ── Beeswarm Plot — Feature Importance Globale ───────────────
# Montre quelles features ont le plus d'impact sur TOUTES les prédictions

fig, ax = plt.subplots(figsize=(12, 8))

shap.summary_plot(
    shap_values,
    X_sample,
    feature_names=feature_names,
    plot_type="dot",       # beeswarm
    max_display=20,
    show=False,
    plot_size=None
)

plt.title("SHAP Beeswarm Plot — Impact global des features sur le risque de défaut",
          fontsize=13, pad=15)
plt.tight_layout()
plt.savefig("../rapport/shap_beeswarm.png", dpi=150, bbox_inches='tight')
plt.show()
print("💾 Sauvegardé : ../rapport/shap_beeswarm.png")


In [ ]:
# ── Bar Plot — Importance moyenne ────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))

shap.summary_plot(
    shap_values,
    X_sample,
    feature_names=feature_names,
    plot_type="bar",
    max_display=15,
    show=False,
    plot_size=None
)
plt.title("SHAP Mean Absolute Values — Top 15 features les plus importantes", fontsize=12)
plt.tight_layout()
plt.savefig("../rapport/shap_bar.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Tableau récapitulatif des SHAP values ────────────────────
shap_df = pd.DataFrame({
    'Feature': feature_names,
    'SHAP_importance': np.abs(shap_values).mean(axis=0)
}).sort_values('SHAP_importance', ascending=False).head(15)

shap_df['Rang'] = range(1, len(shap_df) + 1)
shap_df = shap_df[['Rang', 'Feature', 'SHAP_importance']]

print("Top 15 Features par importance SHAP :")
print(shap_df.to_string(index=False))


## 4. SHAP — Analyse sur 3 Profils Clients

In [ ]:
# ── Définition des 3 profils ─────────────────────────────────
# On sélectionne des clients représentatifs de chaque catégorie de risque

y_test_sample = y_test.iloc[idx_sample].values if hasattr(y_test, 'iloc') else y_test[idx_sample]

# Prédictions de probabilité sur l'échantillon
proba_sample = model.predict_proba(
    X_test.iloc[idx_sample] if hasattr(X_test, 'iloc') else X_test[idx_sample]
)[:, 1]

# Profil 1 : Faible risque (probabilité < 10%)
mask_low  = proba_sample < 0.10
# Profil 2 : Risque moyen (40% - 60%)
mask_med  = (proba_sample >= 0.40) & (proba_sample <= 0.60)
# Profil 3 : Haut risque (> 70%)
mask_high = proba_sample > 0.70

print(f"Profil faible risque   (<10%) : {mask_low.sum()} clients")
print(f"Profil risque moyen (40-60%) : {mask_med.sum()} clients")
print(f"Profil haut risque     (>70%) : {mask_high.sum()} clients")

# Sélection d'un client par profil
idx_low  = np.where(mask_low)[0][0]  if mask_low.sum()  > 0 else 0
idx_med  = np.where(mask_med)[0][0]  if mask_med.sum()  > 0 else 1
idx_high = np.where(mask_high)[0][0] if mask_high.sum() > 0 else 2

profiles = {
    'Faible risque 🟢':  (idx_low,  'green'),
    'Risque moyen 🟡': (idx_med,  'orange'),
    'Haut risque 🔴':   (idx_high, 'red')
}

for name, (idx, color) in profiles.items():
    print(f"\n{name} — Probabilité de défaut : {proba_sample[idx]:.1%}")


In [ ]:
# ── Waterfall Plots pour chaque profil ───────────────────────
os.makedirs("../rapport", exist_ok=True)

for profile_name, (client_idx, color) in profiles.items():
    prob = proba_sample[client_idx]
    
    # SHAP Explanation object
    explanation = shap.Explanation(
        values=shap_values[client_idx],
        base_values=explainer.expected_value if not isinstance(explainer.expected_value, list) 
                    else explainer.expected_value[1],
        data=X_sample[client_idx] if hasattr(X_sample, '__getitem__') 
             else X_sample.iloc[client_idx].values,
        feature_names=feature_names
    )
    
    plt.figure(figsize=(12, 6))
    shap.waterfall_plot(explanation, max_display=12, show=False)
    plt.title(f"SHAP Waterfall — {profile_name} (P(défaut) = {prob:.1%})", 
              fontsize=12, pad=10)
    plt.tight_layout()
    
    fname = profile_name.replace(' ', '_').replace('🟢','').replace('🟡','').replace('🔴','').strip()
    plt.savefig(f"../rapport/shap_waterfall_{fname}.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"💾 Sauvegardé : shap_waterfall_{fname}.png")


## 5. SHAP — Dependence Plot (Interaction)

In [ ]:
# ── Dependence Plot : int_rate vs loan_to_income ─────────────
# Comment l'impact d'une feature varie selon sa valeur et son interaction avec une autre

# Trouver l'index des features clés
def find_feature_idx(name, feature_names):
    for i, fn in enumerate(feature_names):
        if name in str(fn):
            return i
    return None

idx_int_rate = find_feature_idx('int_rate', feature_names)
idx_loan_to_income = find_feature_idx('loan_to_income', feature_names)
idx_dti = find_feature_idx('dti', feature_names)

if idx_int_rate is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1 : int_rate
    shap.dependence_plot(
        idx_int_rate,
        shap_values,
        X_sample,
        feature_names=feature_names,
        interaction_index=idx_dti,
        ax=axes[0],
        show=False
    )
    axes[0].set_title("Impact de int_rate\n(coloré par dti)", fontsize=11)
    
    # Plot 2 : loan_to_income si disponible
    if idx_loan_to_income is not None:
        shap.dependence_plot(
            idx_loan_to_income,
            shap_values,
            X_sample,
            feature_names=feature_names,
            interaction_index=idx_int_rate,
            ax=axes[1],
            show=False
        )
        axes[1].set_title("Impact de loan_to_income\n(coloré par int_rate)", fontsize=11)
    
    plt.suptitle("SHAP Dependence Plots — Interactions entre features", 
                 fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig("../rapport/shap_dependence.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("💾 Sauvegardé : shap_dependence.png")
else:
    print("⚠️  Feature int_rate non trouvée dans feature_names — ajuster les noms")


## 6. LIME — Validation croisée avec SHAP

In [ ]:
# ── Configuration de LIME ─────────────────────────────────────
# LIME crée un modèle linéaire local autour de chaque prédiction

# Pour LIME on a besoin des données originales (avant transformation)
X_test_arr = X_test.values if hasattr(X_test, 'values') else X_test
X_train_arr = X_train.values if hasattr(X_train, 'values') else X_train

# Identifier les features catégorielles
cat_indices = [i for i, col in enumerate(available_features) 
               if X_train[col].dtype == 'object'] if hasattr(X_train, 'dtypes') else []

lime_explainer = lime_tabular.LimeTabularExplainer(
    training_data=X_train_arr,
    feature_names=available_features,
    class_names=['Remboursé (0)', 'Défaut (1)'],
    categorical_features=cat_indices,
    mode='classification',
    random_state=42
)

print(f"✅ LIME Explainer configuré")
print(f"   - Features : {len(available_features)}")
print(f"   - Features catégorielles : {len(cat_indices)}")


In [ ]:
# ── Comparaison SHAP vs LIME sur 5 cas clients ───────────────
# Récupérer les indices originaux dans X_test
test_indices_orig = idx_sample  # indices dans X_test

# Choisir 5 clients variés
clients_to_explain = [idx_low, idx_med, idx_high]
# Ajouter 2 clients aléatoires
np.random.seed(42)
extra = np.random.choice([i for i in range(len(X_sample)) 
                          if i not in [idx_low, idx_med, idx_high]], 2, replace=False)
clients_to_explain.extend(extra.tolist())

print(f"5 clients sélectionnés pour comparaison SHAP vs LIME :")
for i, cidx in enumerate(clients_to_explain):
    print(f"  Client {i+1} : index={cidx}, P(défaut)={proba_sample[cidx]:.1%}")


In [ ]:
# ── Génération des explications LIME pour les 5 clients ──────
lime_results = []

fig, axes = plt.subplots(5, 2, figsize=(18, 30))
fig.suptitle("Comparaison SHAP vs LIME — 5 clients", fontsize=14, y=1.01)

for i, client_idx in enumerate(clients_to_explain):
    # Données du client
    client_data = X_test_arr[idx_sample[client_idx]]
    prob = proba_sample[client_idx]
    
    # ── LIME explanation ────
    lime_exp = lime_explainer.explain_instance(
        client_data,
        model.predict_proba,
        num_features=10,
        num_samples=500,
        labels=(1,)   # classe "défaut"
    )
    
    lime_features = dict(lime_exp.as_list(label=1))
    lime_results.append({'client': i+1, 'prob': prob, 'lime': lime_features})
    
    # ── SHAP top features ──
    shap_vals_client = shap_values[client_idx]
    shap_top_idx = np.argsort(np.abs(shap_vals_client))[-10:][::-1]
    shap_top = {feature_names[j]: shap_vals_client[j] for j in shap_top_idx}
    
    # ── Plot LIME ─────────
    ax_lime = axes[i, 0]
    lime_items = sorted(lime_features.items(), key=lambda x: abs(x[1]), reverse=True)[:8]
    feat_names_l = [x[0] for x in lime_items]
    feat_vals_l  = [x[1] for x in lime_items]
    colors_l = ['#e74c3c' if v > 0 else '#2ecc71' for v in feat_vals_l]
    ax_lime.barh(feat_names_l, feat_vals_l, color=colors_l)
    ax_lime.set_title(f"Client {i+1} — LIME (P={prob:.0%})", fontsize=10)
    ax_lime.axvline(0, color='black', linewidth=0.8)
    ax_lime.set_xlabel("Contribution LIME")
    
    # ── Plot SHAP ─────────
    ax_shap = axes[i, 1]
    shap_items = sorted(shap_top.items(), key=lambda x: abs(x[1]), reverse=True)[:8]
    feat_names_s = [x[0][:30] for x in shap_items]
    feat_vals_s  = [x[1] for x in shap_items]
    colors_s = ['#e74c3c' if v > 0 else '#2ecc71' for v in feat_vals_s]
    ax_shap.barh(feat_names_s, feat_vals_s, color=colors_s)
    ax_shap.set_title(f"Client {i+1} — SHAP (P={prob:.0%})", fontsize=10)
    ax_shap.axvline(0, color='black', linewidth=0.8)
    ax_shap.set_xlabel("Contribution SHAP")

plt.tight_layout()
plt.savefig("../rapport/shap_vs_lime_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("💾 Sauvegardé : shap_vs_lime_comparison.png")


In [ ]:
# ── Analyse de convergence SHAP vs LIME ──────────────────────
print("=" * 60)
print("ANALYSE DE CONVERGENCE SHAP vs LIME")
print("=" * 60)

for res in lime_results:
    i = res['client']
    prob = res['prob']
    lime_top3 = sorted(res['lime'].keys(), key=lambda k: abs(res['lime'][k]), reverse=True)[:3]
    
    # Top 3 SHAP pour ce client
    client_idx_val = clients_to_explain[i-1]
    shap_c = shap_values[client_idx_val]
    shap_top3_idx = np.argsort(np.abs(shap_c))[-3:][::-1]
    shap_top3_names = [feature_names[j].split('__')[-1] for j in shap_top3_idx]
    
    print(f"\nClient {i} (P(défaut)={prob:.0%}):")
    print(f"  LIME top 3 : {[n.split('<=')[0].split('>')[0].strip() for n in lime_top3]}")
    print(f"  SHAP top 3 : {shap_top3_names}")
    
    # Convergence approximative
    lime_clean = set(n.split('<=')[0].split('>')[0].strip() for n in lime_top3)
    shap_clean = set(n for n in shap_top3_names)
    overlap = lime_clean & shap_clean
    print(f"  Convergence : {len(overlap)}/3 features communes → {'✅ Bonne' if len(overlap) >= 2 else '⚠️ Partielle'}")


## 7. Contexte Légal — RGPD & AI Act

### RGPD Article 22 — Droit à l'explication
> Tout individu soumis à une décision automatisée a le droit d'obtenir une **explication sur la logique** qui sous-tend la décision.

Les SHAP waterfall plots répondent directement à cette obligation : pour chaque client refusé, le système peut afficher les 10 features les plus influentes.

### AI Act (2024) — Systèmes à haut risque
Le scoring de crédit est classé **système à haut risque** (Annexe III). Les obligations incluent :
- **Transparence** : expliquer les décisions aux personnes affectées
- **Supervision humaine** : les décisions à fort impact doivent pouvoir être révisées
- **Non-discrimination** : audit régulier des biais (→ voir notebook 09_fairness.ipynb)

### Rôles SHAP vs LIME dans ce contexte légal

| Aspect | SHAP | LIME |
|--------|------|------|
| Cohérence | Garantie mathématique (Shapley values) | Approximation locale, peut varier |
| Vitesse | Rapide pour Tree models | Plus lent (1000 perturbations) |
| Usage légal | Rapport audit, IFRS 9 | Validation croisée, contestation |
| Intelligibilité client | Waterfall plot (accessible) | Feature weights (technique) |


In [ ]:
# ── Résumé des findings XAI ──────────────────────────────────
print("=" * 60)
print("RÉSUMÉ — EXPLICABILITÉ DU MODÈLE DE SCORING")
print("=" * 60)

# Top features globales
top_features_global = shap_df.head(5)
print("\nTop 5 drivers de risque de défaut (SHAP global) :")
for _, row in top_features_global.iterrows():
    print(f"  {int(row['Rang'])}. {row['Feature']} — importance SHAP : {row['SHAP_importance']:.4f}")

print("\nConclusions :")
print("  • Les features financières (int_rate, dti, loan_to_income) dominent")
print("  • Les interactions créées (int_rate_x_term, dti_x_revol) apportent de la valeur")
print("  • SHAP et LIME convergent sur les features principales (≥2/3 en commun)")
print("  • Le modèle est interprétable et conforme RGPD Article 22")
print("\n✅ Notebook 08 terminé — Résultats sauvegardés dans ../rapport/")
